In [94]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rohanharode07/webmd-drug-reviews-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/hungnguyen/.cache/kagglehub/datasets/rohanharode07/webmd-drug-reviews-dataset/versions/1


In [95]:
# List the head of dataset files
import os
print("Files in dataset directory:", os.listdir(path))
# Load the dataset using pandas
import pandas as pd
dataset_file = os.path.join(path, "webmd.csv")
df = pd.read_csv(dataset_file)
df.head()

Files in dataset directory: ['webmd.csv']


,Age,Condition,Date,Drug,DrugId,EaseofUse,Effectiveness,Reviews,Satisfaction,Sex,Sides,UsefulCount
0,75 or over,Stuffy Nose,9/21/2014,25dph-7.5peh,146724,5,5,I'm a retired physician and of all the meds I ...,5,Male,"Drowsiness, dizziness , dry mouth /nose/thro...",0
1,25-34,Cold Symptoms,1/13/2011,25dph-7.5peh,146724,5,5,cleared me right up even with my throat hurtin...,5,Female,"Drowsiness, dizziness , dry mouth /nose/thro...",1
2,65-74,Other,7/16/2012,warfarin (bulk) 100 % powder,144731,2,3,why did my PTINR go from a normal of 2.5 to ov...,3,Female,,0
3,75 or over,Other,9/23/2010,warfarin (bulk) 100 % powder,144731,2,2,FALLING AND DON'T REALISE IT,1,Female,,0
4,35-44,Other,1/6/2009,warfarin (bulk) 100 % powder,144731,1,1,My grandfather was prescribed this medication ...,1,Male,,1


In [ ]:
# get 1000 rows from df 
# df = df.sample(10000)
# print to the new file called webmd_1000.csv
# df.to_csv("webmd_1000.csv", index=False)


# df = df[["Age", "Condition", "Drug", "DrugId", "Satisfaction", "Sex", "Reviews"]]

df.isnull().sum()

Age              0
Condition        0
Date             0
Drug             0
DrugId           0
EaseofUse        0
Effectiveness    0
Reviews          1
Satisfaction     0
Sex              0
Sides            0
UsefulCount      0
dtype: int64

In [97]:
df = df.dropna()

for col in df.columns:
    if df[col].dtype.kind == "O":
        df[col] = df[col].str.strip()


def satisfaction_to_class(rating: int) -> int:
    """Map WebMD 1–5 satisfaction to 0=Negative, 1=Neutral, 2=Positive."""
    r = int(rating)
    if r <= 2:
        return 0
    if r == 3:
        return 1
    return 2

def filter_anomalies(df: pd.DataFrame, min_review_len: int = 10) -> pd.DataFrame:
    """Remove nulls and very short/uninformative reviews."""
    before = len(df)
    df = df.dropna(subset=["Reviews"]).copy()
    df = df[df["Reviews"].str.strip().str.len() >= min_review_len].copy()
    print(f"\n[3] Anomaly filter: removed {before - len(df)} rows "
          f"(null or < {min_review_len} chars).  Remaining: {len(df):,}")
    return df
 

In [98]:
# Negation words — preserve even after stopword removal
NEGATIONS = {
    "no", "not", "nor", "never", "neither", "hardly", "barely",
    "scarcely", "without", "cannot", "didn't", "doesn't", "wasn't",
    "shouldn't", "wouldn't", "couldn't", "won't", "can't", "don't",
}
 
CONTRACTIONS = {
    r"n't":   " not",
    r"'re":   " are",
    r"'s":    " is",
    r"'d":    " would",
    r"'ll":   " will",
    r"'ve":   " have",
    r"'m":    " am",
    r"won't": "will not",
    r"can't": "cannot",
}

import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer
    nltk.download("stopwords", quiet=True)
    nltk.download("punkt", quiet=True)
    STOPWORDS = set(stopwords.words("english"))
    STEMMER = PorterStemmer()
    HAS_NLTK = True
except ImportError:
    # Fallback: compact built-in stopword list
    STOPWORDS = {
        "i", "me", "my", "we", "our", "you", "your", "he", "she", "it", "its", "they", "them",
        "what", "which", "who", "this", "that", "these", "those", "am", "is", "are", "was",
        "were", "be", "been", "being", "have", "has", "had", "do", "does", "did", "doing",
        "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of",
        "at", "by", "for", "with", "about", "against", "between", "into", "through", "during",
        "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on",
        "off", "over", "under", "then", "once", "here", "there", "when", "where", "why", "how",
        "all", "each", "few", "more", "most", "other", "some", "such", "than", "too", "very",
        "just", "can", "will", "should", "now", "s", "t", "re", "ve", "ll", "d", "m",
    }
    STEMMER = None
    HAS_NLTK = False

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    VADER = SentimentIntensityAnalyzer()
    HAS_VADER = True
except ImportError:
    VADER = None
    HAS_VADER = False

try:
    from imblearn.over_sampling import SMOTE  # noqa: F401
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False

In [99]:
import unicodedata
import re

def clean_text(text: str,
               remove_stopwords: bool = True,
               stem: bool = False) -> str:
    """
    Full text cleaning pipeline:
      - Lowercase
      - Normalise unicode (accents -> ASCII)
      - Remove HTML tags
      - Expand common contractions
      - Remove URLs and emails
      - Remove punctuation
      - Remove standalone digits
      - Tokenise on whitespace
      - Remove stopwords (preserve negations)
      - Optional Porter stemming
      - Drop single-character tokens
    """
    if not isinstance(text, str):
        return ""
 
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"<[^>]+>", " ", text)                    # strip HTML
 
    for pattern, replacement in CONTRACTIONS.items():
        text = re.sub(pattern, replacement, text)
 
    text = re.sub(r"http\S+|www\.\S+|\S+@\S+", " ", text)   # URLs / emails
    text = re.sub(r"[^\w\s]", " ", text)                    # punctuation
    text = re.sub(r"\b\d+\b", " ", text)                    # standalone digits
 
    tokens = text.split()
 
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS or t in NEGATIONS]
 
    if stem and STEMMER is not None:
        tokens = [STEMMER.stem(t) for t in tokens]
 
    tokens = [t for t in tokens if len(t) > 1 or t == "i"]
 
    return " ".join(tokens)

## Machine learning pipeline

The cells below map WebMD 1–5 satisfaction to three classes (negative / neutral / positive), clean review text, hold out a test set, then train and evaluate **Multinomial NB**, **Random Forest + TF-IDF**, and an **LSTM**. Helpers are documented in code; long-running cells are marked.

In [100]:
import re

import altair as alt
import numpy as np
from IPython.display import display
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

def confusion_matrix_altair(y_true, y_pred, labels=None):
    """Confusion matrix as an Altair heatmap (counts in cells)."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    if labels is None:
        labels = sorted(np.unique(np.concatenate([y_true.ravel(), y_pred.ravel()])).tolist())
    rows = [
        {"True": str(labels[i]), "Predicted": str(labels[j]), "Count": int(cm[i, j])}
        for i in range(len(labels))
        for j in range(len(labels))
    ]
    src = pd.DataFrame(rows)
    base = alt.Chart(src).encode(
        x=alt.X("Predicted:O", title="Predicted"),
        y=alt.Y("True:O", title="True"),
    )
    heat = base.mark_rect().encode(
        color=alt.Color("Count:Q", scale=alt.Scale(scheme="blues")),
        tooltip=["True", "Predicted", "Count"],
    ).properties(width=320, height=280, title="Confusion matrix")
    text = base.mark_text(baseline="middle", fontSize=11).encode(text="Count:Q")
    display(heat + text)


# Modeling frame: encoded labels + cleaned text (tokenizer/vectorizer fit only on train later).
mldf = df.copy()
mldf["Satisfaction"] = mldf["Satisfaction"].astype(int).map(satisfaction_to_class)
mldf["Reviews"] = mldf["Reviews"].apply(clean_text)
mldf = filter_anomalies(mldf)

train_index, test_index = train_test_split(
    mldf.index, test_size=0.25, random_state=0, shuffle=True
)
train_set = mldf.loc[train_index]
test_set = mldf.loc[test_index]
print("train_set:", train_set.shape, "test_set:", test_set.shape)


[3] Anomaly filter: removed 1287 rows (null or < 10 chars).  Remaining: 8,712
train_set: (6534, 12) test_set: (2178, 12)


### Reusable trainers and evaluation

The next cell defines `report_classification`, `train_multinomial_nb`, `train_random_forest_tfidf`, and LSTM helpers. Sklearn models use **sparse** matrices (no `.toarray()`). The LSTM uses `LabelEncoder` and `sparse_categorical_crossentropy` so class indices stay consistent with `classification_report`.

In [101]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer


def report_classification(
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    *,
    target_names=None,
    plot_confusion=True,
):
    """Train/test accuracy, macro F1, classification report, optional confusion heatmap."""
    y_pred_tr = model.predict(X_train)
    y_pred_te = model.predict(X_test)
    print(f"\nAccuracy (train): {accuracy_score(y_train, y_pred_tr):.4f}")
    print(f"Accuracy (test):  {accuracy_score(y_test, y_pred_te):.4f}")
    print(f"F1 macro (test):  {f1_score(y_test, y_pred_te, average='macro'):.4f}\n")
    print(classification_report(y_test, y_pred_te, target_names=target_names, digits=2))
    if plot_confusion:
        confusion_matrix_altair(y_test, y_pred_te)


def train_multinomial_nb(
    train_df,
    test_df,
    *,
    drop_neutral=False,
    max_features=2500,
    min_df=10,
    max_df=0.8,
    alpha=1.0,
):
    """Bag-of-words (sparse) + Multinomial Naive Bayes. Optionally drop Satisfaction == 1 (neutral)."""
    train_df = train_df.copy()
    test_df = test_df.copy()
    if drop_neutral:
        train_df = train_df[train_df["Satisfaction"] != 1]
        test_df = test_df[test_df["Satisfaction"] != 1]
    vectorizer = CountVectorizer(max_features=max_features, min_df=min_df, max_df=max_df)
    X_train = vectorizer.fit_transform(train_df["Reviews"])
    X_test = vectorizer.transform(test_df["Reviews"])
    y_train = train_df["Satisfaction"].to_numpy()
    y_test = test_df["Satisfaction"].to_numpy()
    model = MultinomialNB(alpha=alpha)
    model.fit(X_train, y_train)
    classes = sorted(pd.unique(np.concatenate([y_train, y_test])))
    target_names = [str(c) for c in classes]
    report_classification(model, X_train, X_test, y_train, y_test, target_names=target_names)
    return model, vectorizer


def train_random_forest_tfidf(
    train_df,
    test_df,
    *,
    drop_neutral=False,
    max_features=2500,
    min_df=10,
    max_df=0.8,
    min_samples_split=6,
    random_state=0,
    n_jobs=-1,
    n_estimators=100,
):
    """TF-IDF (sparse) + RandomForest. Lower n_estimators for faster runs during development."""
    train_df = train_df.copy()
    test_df = test_df.copy()
    if drop_neutral:
        train_df = train_df[train_df["Satisfaction"] != 1]
        test_df = test_df[test_df["Satisfaction"] != 1]
    vectorizer = TfidfVectorizer(max_features=max_features, min_df=min_df, max_df=max_df)
    X_train = vectorizer.fit_transform(train_df["Reviews"])
    X_test = vectorizer.transform(test_df["Reviews"])
    y_train = train_df["Satisfaction"].to_numpy()
    y_test = test_df["Satisfaction"].to_numpy()
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        min_samples_split=min_samples_split,
        random_state=random_state,
        n_jobs=n_jobs,
    )
    model.fit(X_train, y_train)
    classes = sorted(pd.unique(np.concatenate([y_train, y_test])))
    target_names = [str(c) for c in classes]
    report_classification(model, X_train, X_test, y_train, y_test, target_names=target_names)
    return model, vectorizer


def build_lstm_binary(num_words=2500, maxlen=200, embedding_dim=100, spatial_dropout=0.2):
    """Embedding -> SpatialDropout1D -> LSTM -> Dense softmax for 2 classes (encoded 0/1)."""
    model = Sequential(
        [
            Embedding(num_words, embedding_dim, input_length=maxlen),
            SpatialDropout1D(spatial_dropout),
            LSTM(embedding_dim),
            Dense(2, activation="softmax"),
        ]
    )
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def train_lstm_sentiment(
    train_reviews,
    test_reviews,
    y_train_raw,
    y_test_raw,
    *,
    num_words=2500,
    maxlen=200,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
):
    """
    Tokenizer fit only on training reviews. y_* must be original labels {0, 2} after dropping neutral.
    Returns model, tokenizer, label_encoder, history.
    """
    le = LabelEncoder()
    y_train = le.fit_transform(y_train_raw)
    y_test = le.transform(y_test_raw)
    tokenizer = Tokenizer(num_words=num_words, split=" ", lower=False)
    tokenizer.fit_on_texts(train_reviews)
    X_train = pad_sequences(tokenizer.texts_to_sequences(train_reviews), maxlen=maxlen)
    X_test = pad_sequences(tokenizer.texts_to_sequences(test_reviews), maxlen=maxlen)
    model = build_lstm_binary(num_words=num_words, maxlen=maxlen)
    history = model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=2,
                min_delta=1e-4,
                restore_best_weights=True,
            )
        ],
        verbose=1,
    )
    return model, tokenizer, le, history, X_train, X_test, y_train, y_test


def evaluate_keras_binary(model, X_test, y_test_int, *, batch_size=64):
    """Test loss/accuracy and sklearn report (encoded labels 0/1 with names Negative/Positive)."""
    loss, acc = model.evaluate(X_test, y_test_int, batch_size=batch_size, verbose=0)
    print(f"\nLoss (test): {loss:.4f}\nAccuracy (test): {acc:.4f}\n")
    proba = model.predict(X_test, batch_size=batch_size, verbose=0)
    pred = np.argmax(proba, axis=-1)
    target_names = ["Negative", "Positive"]
    print(classification_report(y_test_int, pred, target_names=target_names, digits=2))
    confusion_matrix_altair(y_test_int, pred, labels=[0, 1])


def predict_sentiment(text, model, tokenizer, maxlen, label_encoder):
    """clean_review -> sequence -> pad -> predict -> 'Negative' or 'Positive' (WebMD codes 0 / 2)."""
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=maxlen)
    idx = int(np.argmax(model.predict(padded, verbose=0), axis=-1))
    original = int(label_encoder.inverse_transform([idx])[0])
    return "Positive" if original == 2 else "Negative"

### Multinomial Naive Bayes + `CountVectorizer`

Two runs: **3-class** (all satisfaction codes) and **binary** (drop neutral `Satisfaction == 1`).

In [102]:
%%time
nb_3class, _ = train_multinomial_nb(train_set, test_set, drop_neutral=False)


Accuracy (train): 0.7300
Accuracy (test):  0.6529
F1 macro (test):  0.5421

              precision    recall  f1-score   support

           0       0.67      0.70      0.69       872
           1       0.34      0.17      0.23       295
           2       0.68      0.75      0.71      1011

    accuracy                           0.65      2178
   macro avg       0.56      0.54      0.54      2178
weighted avg       0.63      0.65      0.64      2178



alt.LayerChart(...)

CPU times: user 130 ms, sys: 5.9 ms, total: 136 ms
Wall time: 148 ms


In [103]:
%%time
nb_binary, _ = train_multinomial_nb(train_set, test_set, drop_neutral=True)


Accuracy (train): 0.8090
Accuracy (test):  0.7647
F1 macro (test):  0.7628

              precision    recall  f1-score   support

           0       0.75      0.73      0.74       872
           2       0.77      0.80      0.78      1011

    accuracy                           0.76      1883
   macro avg       0.76      0.76      0.76      1883
weighted avg       0.76      0.76      0.76      1883



alt.LayerChart(...)

CPU times: user 125 ms, sys: 3.95 ms, total: 129 ms
Wall time: 135 ms


### Random Forest + `TfidfVectorizer`

**Note:** Full `n_estimators=100` on ~220k×2500 sparse features can take **tens of minutes**. For a quicker check, rerun `train_random_forest_tfidf(..., n_estimators=25)` in a scratch cell or change the default in the helper cell above.

In [104]:
%%time
rf_3class, _ = train_random_forest_tfidf(train_set, test_set, drop_neutral=False)


Accuracy (train): 0.9957
Accuracy (test):  0.6745
F1 macro (test):  0.4951

              precision    recall  f1-score   support

           0       0.68      0.73      0.70       872
           1       0.78      0.02      0.05       295
           2       0.67      0.82      0.74      1011

    accuracy                           0.67      2178
   macro avg       0.71      0.52      0.50      2178
weighted avg       0.69      0.67      0.63      2178



alt.LayerChart(...)

CPU times: user 2.59 s, sys: 77.1 ms, total: 2.66 s
Wall time: 668 ms


In [105]:
%%time
rf_binary, _ = train_random_forest_tfidf(train_set, test_set, drop_neutral=True)


Accuracy (train): 0.9979
Accuracy (test):  0.7727
F1 macro (test):  0.7702

              precision    recall  f1-score   support

           0       0.77      0.72      0.75       872
           2       0.77      0.82      0.79      1011

    accuracy                           0.77      1883
   macro avg       0.77      0.77      0.77      1883
weighted avg       0.77      0.77      0.77      1883



alt.LayerChart(...)

CPU times: user 1.95 s, sys: 82.3 ms, total: 2.03 s
Wall time: 520 ms


### LSTM (binary: negative vs positive)

Tokenizer is fit on **training** reviews only. `maxlen=200` and `num_words=2500` match the vectorizer caps above for a fair-ish comparison.

In [106]:
# %%time
# LSTM_MAXLEN = 200
# tr_bin = train_set[train_set["Satisfaction"] != 1]
# te_bin = test_set[test_set["Satisfaction"] != 1]
# lstm_model, lstm_tokenizer, le_lstm, lstm_history, _, X_test_lstm, _, y_test_lstm = train_lstm_sentiment(
#     tr_bin["Reviews"].to_numpy(),
#     te_bin["Reviews"].to_numpy(),
#     tr_bin["Satisfaction"].to_numpy(),
#     te_bin["Satisfaction"].to_numpy(),
#     maxlen=LSTM_MAXLEN,
# )

In [107]:
# evaluate_keras_binary(lstm_model, X_test_lstm, y_test_lstm)

In [108]:
# predict_sentiment(
#     "The drug is expensive but it is worth every cent.",
#     lstm_model,
#     lstm_tokenizer,
#     LSTM_MAXLEN,
#     le_lstm,
# )

### Transformer: DistilBERT or BioBERT (PyTorch + Hugging Face)

Fine-tune **[`distilbert-base-uncased`](https://huggingface.co/distilbert-base-uncased)** (fast, general English) or **[`dmis-lab/biobert-base-cased-v1.2`](https://huggingface.co/dmis-lab/biobert-base-cased-v1.2)** (biomedical vocabulary) for **3-class** sentiment (negative / neutral / positive), aligned with `train_set` / `test_set`.

Uses **raw** `Reviews` from `df` (not `clean_text`), which is better for subword models. Set `TRANSFORMER_MAX_TRAIN_SAMPLES` to an integer for a quicker run; `None` uses the full training split.

**Dependencies:** `pip install transformers datasets accelerate` (PyTorch must match your machine).


In [110]:
import inspect

import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# --- choose backbone ---
TRANSFORMER_BACKBONE = "distilbert"  # or "biobert"
PRETRAINED = {
    "distilbert": "distilbert-base-uncased",
    "biobert": "dmis-lab/biobert-base-cased-v1.2",
}[TRANSFORMER_BACKBONE]

NUM_EPOCHS = 3
TRANSFORMER_MAX_TRAIN_SAMPLES = 10000  # e.g. 12_000 for a faster experiment; None = all train rows
SEED = 0
MAX_LENGTH = 128
OUTPUT_DIR = "./hf-sentiment-checkpoints"

# Same labels as the rest of the notebook: 0=negative, 1=neutral, 2=positive
TARGET_NAMES = ["Negative", "Neutral", "Positive"]
NUM_LABELS = 3


def _reviews_for_indices(indices):
    s = df.loc[indices, "Reviews"].fillna("").astype(str).str.strip()
    return s.tolist()


train_texts = _reviews_for_indices(train_set.index)
test_texts = _reviews_for_indices(test_set.index)
y_train = train_set["Satisfaction"].to_numpy(dtype=np.int64)
y_test = test_set["Satisfaction"].to_numpy(dtype=np.int64)

if TRANSFORMER_MAX_TRAIN_SAMPLES is not None and TRANSFORMER_MAX_TRAIN_SAMPLES < len(train_texts):
    rng = np.random.default_rng(SEED)
    pick = rng.choice(len(train_texts), size=TRANSFORMER_MAX_TRAIN_SAMPLES, replace=False)
    train_texts = [train_texts[i] for i in pick]
    y_train = y_train[pick]
    print(f"Using subsampled training set: {len(train_texts):,} rows")
else:
    print(f"Training on full train split: {len(train_texts):,} rows")

raw_train = Dataset.from_dict({"text": train_texts, "label": y_train.tolist()})
raw_eval = Dataset.from_dict({"text": test_texts, "label": y_test.tolist()})

tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)


def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


train_tok = raw_train.map(tokenize_batch, batched=True, remove_columns=["text"])
eval_tok = raw_eval.map(tokenize_batch, batched=True, remove_columns=["text"])

model = AutoModelForSequenceClassification.from_pretrained(PRETRAINED, num_labels=NUM_LABELS)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, pred)),
        "f1_macro": float(f1_score(labels, pred, average="macro")),
    }


# TrainingArguments API differs slightly across transformers versions
_ta_params = inspect.signature(TrainingArguments.__init__).parameters
_use_eval_strategy = "eval_strategy" in _ta_params

_common_kw = dict(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.06,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=200,
    report_to="none",
    seed=SEED,
)
if _use_eval_strategy:
    training_args = TrainingArguments(
        **_common_kw,
        eval_strategy="epoch",
        save_strategy="epoch",
    )
else:
    training_args = TrainingArguments(
        **_common_kw,
        evaluation_strategy="epoch",
        save_strategy="epoch",
    )

_trainer_kw = dict(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer = Trainer(**_trainer_kw, processing_class=tokenizer)
else:
    trainer = Trainer(**_trainer_kw, tokenizer=tokenizer)

trainer.train()

# Avoid trainer.evaluate()/predict() notebook callback bug in some transformers versions.
collator = DataCollatorWithPadding(tokenizer)
eval_for_infer = eval_tok.with_format("python")
eval_loader = torch.utils.data.DataLoader(
    eval_for_infer,
    batch_size=training_args.per_device_eval_batch_size,
    shuffle=False,
    collate_fn=collator,
)

model.eval()
all_logits = []
all_labels = []
with torch.no_grad():
    for batch in eval_loader:
        labels = batch["labels"].cpu().numpy()
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits.detach().cpu().numpy()
        all_labels.append(labels)
        all_logits.append(logits)

y_true = np.concatenate(all_labels)
y_pred = np.argmax(np.concatenate(all_logits, axis=0), axis=-1)

print(f"\nAccuracy (test): {accuracy_score(y_true, y_pred):.4f}")
print(f"F1 macro (test): {f1_score(y_true, y_pred, average='macro'):.4f}\n")
print(classification_report(y_true, y_pred, target_names=TARGET_NAMES, digits=2))
confusion_matrix_altair(y_true, y_pred, labels=[0, 1, 2])


Training on full train split: 6,534 rows


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6934.91it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/Users/hungnguyen/study - master IT 2024/Term 4/research/research-code/venv/lib/p

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 